<a href="https://www.kaggle.com/code/shrikantlala/rezno-ai-backend?scriptVersionId=371095879" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
!pip install -U langchain-google-genai
!pip install arxiv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.9/588.9 kB 6.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 12.0 MB/s  0:00:00
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.49.0
    Uninstalling google-auth-2.49.0:
      Successfully uninstalled google-auth-2.49.0
  Attempting uninstall: langchain-core━━━━━━━━━━ 0/4 [google-auth]
    Found existing installation: langchain-core 1.7.0m0/4 [google-auth]
    Uninstalling langchain-core-1.7.0:━━━━━━ 0/4 [google-auth]
      Successfully uninstalled langchain-core-1.7.0━━━━━━━━━━━━━━━ 1/4 [langchain-core]
  Attempting uninstall: google-genaim━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/4 [langchain-core]
    Found existing installation: google-genai 2.12.1━━━━━━━━━━ 1/4 [langchain-core]
    Uninstalling google-genai-2.12.1:╺━━━━━━━━━━━━━━━━━━━ 2/4 [google-genai]
      Successfully uninstalled google-genai-2.12.1━━━━━━━━━━━━━━━━ 2/4 [google-genai]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

from IPython.display import display, Markdown
from dotenv import load_dotenv
import os

from pydantic import BaseModel, Field

import arxiv

In [3]:
load_dotenv()

False

In [4]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("GOOGLE_API_KEY")


In [5]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0.7,
    google_api_key=secret_value_0
)

> alright let's define types first
---
1. arxiv_papers



In [6]:
# client = arxiv.Client
# client

def arxiv_search(query: str, max_results=10):

    search = arxiv.Search(
        query=query,
        max_results=max_results,
        sort_by=arxiv.SortCriterion.Relevance
    )

    client = arxiv.Client()

    papers = []

    for paper in client.results(search):

        papers.append({
            "id": paper.entry_id,
            "title": paper.title,
            "abstract": paper.summary,
            "authors": [a.name for a in paper.authors],
            "published": paper.published,
            "categories": paper.categories,
            "pdf_url": paper.pdf_url
        })

    return papers

In [7]:
class ResearchQueries(BaseModel):
    queries: list[str] = Field(
        description="4 to 10 relevant research search queries"
    )

In [8]:
def subquery_generater(user_input : str, sub_queries=3):

    prompt = ChatPromptTemplate.from_messages([
        ("system","You are an expert researcher and you know How a given concept/topic/theory of any given topic is connected to another. and you have ability to connect based on relevance and likelihood of occurence of them together"),
        ("human","given a user query '{user_input}', give me {sub_queries} no. of sub_queries which are connected with the main query in such way.")
    ])

    structured_llm = llm.with_structured_output(
        ResearchQueries,
        method="json_schema"
    )

    chain = prompt | structured_llm

    return chain.invoke({
        "user_input" : user_input,
        "sub_queries" : sub_queries
    })

In [9]:
result = subquery_generater(
    "Gradient Descent",
    sub_queries=5
)

query_list = result.queries

for query in query_list:
    papers = arxiv_search(query)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3940: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


In [10]:
query_list

['Stochastic Gradient Descent optimization',
 'Learning rate tuning in Gradient Descent',
 'Gradient Descent vs Adam Optimizer',
 'Convergence issues in Gradient Descent',
 'Batch Gradient Descent mathematics']

In [11]:
for i in papers:
    print(f"* {i['title']}")
    print("\n")

* Beyond NTK with Vanilla Gradient Descent: A Mean-Field Analysis of Neural Networks with Polynomial Width, Samples, and Time


* Quadratic number of nodes is sufficient to learn a dataset via gradient descent


* The Parallel Knowledge Gradient Method for Batch Bayesian Optimization


* Continuous Delayed-Memory Stochastic Gradient Descent and Continuous-Time Reinforcement Learning from History of Astrophysical Time Series Studies


* Energy-performance tradeoffs in server farms with batch services and setup times


* Stochastic Gradient Descent for Barycenters in Wasserstein Space


* Generalized Deep Learning-based Proximal Gradient Descent for MR Reconstruction


* High-Probability Guarantees for Random Zeroth-Order (Stochastic) Gradient Descent


* Gradient Descent with Polyak's Momentum Finds Flatter Minima via Large Catapults


* MBGDT:Robust Mini-Batch Gradient Descent




In [12]:
papers

[{'id': 'http://arxiv.org/abs/2306.16361v2',
  'title': 'Beyond NTK with Vanilla Gradient Descent: A Mean-Field Analysis of Neural Networks with Polynomial Width, Samples, and Time',
  'abstract': 'Despite recent theoretical progress on the non-convex optimization of two-layer neural networks, it is still an open question whether gradient descent on neural networks without unnatural modifications can achieve better sample complexity than kernel methods. This paper provides a clean mean-field analysis of projected gradient flow on polynomial-width two-layer neural networks. Different from prior works, our analysis does not require unnatural modifications of the optimization algorithm. We prove that with sample size $n = O(d^{3.1})$ where $d$ is the dimension of the inputs, the network trained with projected gradient flow converges in $\\text{poly}(d)$ time to a non-trivial error that is not achievable by kernel methods using $n \\ll d^4$ samples, hence demonstrating a clear separation b

# version 2